In [1]:
import os
import math
import pandas as pd
import numpy as np
import osmnx as ox
import networkx as nx
from pathlib import Path

GRAPH_XML = Path(r"new.osm")

# Load OSM graph (walkable)
g = ox.graph_from_xml(filepath=str(GRAPH_XML), bidirectional=True)
g = ox.convert.to_undirected(g)
print(f"Graph loaded with {g.number_of_nodes()} nodes, {g.number_of_edges()} edges")

# ── TEST 1: Is the graph fully connected? ──────────────────────────────────────
components = list(nx.connected_components(g))
print(f"\nNumber of connected components: {len(components)}")
# If > 1, you have isolated subgraphs — tunnel is likely still disconnected

for i, comp in enumerate(components):
    print(f"  Component {i+1}: {len(comp)} nodes")

# ── TEST 2: Route ACROSS the tunnel ───────────────────────────────────────────
# Pick a point on شارع جمال عبد الناصر (west side)
origin_coords      = (31.22014, 29.942573)   # (lat, lon) — adjust to your area
destination_coords = (31.219085, 29.943421)   # (lat, lon) on شارع توت عنخ امون

orig_node = ox.distance.nearest_nodes(g, origin_coords[1], origin_coords[0])
dest_node = ox.distance.nearest_nodes(g, destination_coords[1], destination_coords[0])

print(f"\nOrigin node:      {orig_node}")
print(f"Destination node: {dest_node}")

try:
    path = nx.shortest_path(g, orig_node, dest_node, weight="length")
    print(f"\n✅ Route found! {len(path)} nodes in path")
    print(f"   Path: {path}")
except nx.NetworkXNoPath:
    print("\n❌ No path found — tunnel is still NOT connected!")
except nx.NodeNotFound as e:
    print(f"\n⚠️  Node not found: {e}")

# ── TEST 3: Visualize the graph (optional) ────────────────────────────────────
# Highlights disconnected components in different colors
if len(components) > 1:
    print("\nVisualizing components (each color = isolated subgraph)...")
    node_colors = []
    color_list = ["red", "blue", "green", "orange", "purple"]
    node_to_comp = {}
    for i, comp in enumerate(components):
        for node in comp:
            node_to_comp[node] = i
    node_colors = [color_list[node_to_comp[n] % len(color_list)] for n in g.nodes()]
    fig, ax = ox.plot_graph(g, node_color=node_colors, node_size=10, show=True)

Graph loaded with 45785 nodes, 65856 edges

Number of connected components: 1
  Component 1: 45785 nodes

Origin node:      4166054659
Destination node: 1886821204

✅ Route found! 7 nodes in path
   Path: [4166054659, 4326474663, 11462717726, 11462717729, 6976044920, 6976044926, 1886821204]


In [8]:
import folium

# get node coordinates
route_nodes = [g.nodes[n] for n in path]
coords = [(n['y'], n['x']) for n in route_nodes]

# create map centered on route
m = folium.Map(location=coords[0], zoom_start=16)
folium.PolyLine(coords, color='red', weight=5).add_to(m)

m.save("route.html")
print("Open route.html in your browser")

Open route.html in your browser


In [13]:

# ── TEST 2: Route ACROSS the tunnel ───────────────────────────────────────────
# Pick a point on شارع جمال عبد الناصر (west side)
destination_coords = (31.221696, 29.944334)   # (lat, lon) — adjust to your area
origin_coords = (31.218448, 29.942487)   # (lat, lon) on شارع توت عنخ امون

orig_node = ox.distance.nearest_nodes(g, origin_coords[1], origin_coords[0])
dest_node = ox.distance.nearest_nodes(g, destination_coords[1], destination_coords[0])

print(f"\nOrigin node:      {orig_node}")
print(f"Destination node: {dest_node}")

try:
    path = nx.shortest_path(g, orig_node, dest_node, weight="length")
    print(f"\n✅ Route found! {len(path)} nodes in path")
    print(f"   Path: {path}")
except nx.NetworkXNoPath:
    print("\n❌ No path found — tunnel is still NOT connected!")
except nx.NodeNotFound as e:
    print(f"\n⚠️  Node not found: {e}")

# ── TEST 3: Visualize the graph (optional) ────────────────────────────────────
# Highlights disconnected components in different colors
if len(components) > 1:
    print("\nVisualizing components (each color = isolated subgraph)...")
    node_colors = []
    color_list = ["red", "blue", "green", "orange", "purple"]
    node_to_comp = {}
    for i, comp in enumerate(components):
        for node in comp:
            node_to_comp[node] = i
    node_colors = [color_list[node_to_comp[n] % len(color_list)] for n in g.nodes()]
    fig, ax = ox.plot_graph(g, node_color=node_colors, node_size=10, show=True)


Origin node:      11462532250
Destination node: 4166055227

✅ Route found! 9 nodes in path
   Path: [11462532250, 1886821204, 6976044926, 6976044920, 11462717729, 11462717726, 4326474663, 4166054659, 4166055227]


In [14]:
import folium

# get node coordinates
route_nodes = [g.nodes[n] for n in path]
coords = [(n['y'], n['x']) for n in route_nodes]

# create map centered on route
m = folium.Map(location=coords[0], zoom_start=16)
folium.PolyLine(coords, color='red', weight=5).add_to(m)

m.save("route.html")
print("Open route.html in your browser")

Open route.html in your browser
